# Project 1: Numerov's method

In this project you will use Numerov's method to solve the stationary Schrödinger equation numerically. Numerov's method solves second-order ODEs of the form

\begin{equation}
  \frac{d^2 y}{dx^2} = f(x) + g(x)\, y(x)
\end{equation}

Because there is no first-derivative term and $y(x)$ only enters linearly on the right-hand side, one can build a very simple, very accurate (fifth-order) method out of this equation, see the lecture slides for the derivation. Discretizing the $x$ axis into points $\{x_0, x_1, \dots\}$ and writing $y_n = y(x_n)$, $f_n = f(x_n)$, $g_n = g(x_n)$, Numerov's method propagates the solution with the recursion

\begin{equation}
  y_{n+1} = \frac{2y_n - y_{n-1} + \frac{h^2}{12} \Big(f_{n+1} + 10 \,
  (f_n + g_n y_n) + f_{n-1} + g_{n-1}y_{n-1}\Big)}{1 - \frac{g_{n+1} h^2}{12}} + O(h^6)
\end{equation}

The same recursion also works backwards, propagating from $x_N$ down to $x_0$ (this will be useful in Part 3).


In [ ]:
# Common imports
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from collections.abc import Callable

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#4a3aa7", "#e34948"]
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (6.5, 4.2),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.edgecolor": "#898781",
    "axes.linewidth": 0.9,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "legend.frameon": False,
    "lines.linewidth": 2.0,
})

## Part 1: Implement Numerov's method

Write a function with the following signature:

```python
def numerov(
    x: np.ndarray,
    y: np.ndarray,
    f: Callable[[float], float],
    g: Callable[[float], float],
    y0: float,
    y1: float,
) -> None:
    ...
```

`x` is a `numpy.ndarray` containing the discretized $x$ points, and `y` is a `numpy.ndarray`, originally empty, that the function fills in place with the solution. `f` and `g` are the two functions defined above, and `y0`, `y1` are the first two values needed to start the recursion (in Python there is no trouble passing a function as an argument to another function).

Test your implementation on the simple case $f(x)=0$, $g(x)=-1$, whose general solution is

\begin{equation}
  y(x) = A \cos( x + \phi )
\end{equation}

You can check a specific solution by choosing

\begin{equation}
  y_0 = A \cos( x_0 + \phi ) \qquad y_1 = A \cos ( x_0 + h + \phi )
\end{equation}

where $x_0$ is the first discretized point and $h$ the step size. Then investigate how sensitive the result is to the choice of $h$: does the error shrink the way you would expect from a fifth-order method?


## Solution

Here is one possible implementation of the `numerov` function:

In [ ]:
def numerov(
    x: np.ndarray,
    y: np.ndarray,
    f: Callable[[float], float],
    g: Callable[[float], float],
    y0: float,
    y1: float,
) -> None:
    """Solve y'' = f(x) + g(x) y(x) with Numerov's method, filling `y` in place."""
    
    y[0], y[1] = y0, y1

    h = x[1] - x[0]
    b = h**2 / 12.0
    f_vals = np.vectorize(f)(x)  # turn the scalar functions f, g into arrays,
    g_vals = np.vectorize(g)(x)  # evaluated once and for all on the grid

    for i in range(1, len(y) - 1):
        y[i + 1] = (
            2 * y[i] - y[i - 1]
            + b * (f_vals[i + 1] + 10 * (f_vals[i] + g_vals[i] * y[i])
                   + f_vals[i - 1] + g_vals[i - 1] * y[i - 1])
        ) / (1 - g_vals[i + 1] * b)


Let's check this on the simple case $g(x) = -1$, $f(x) = 0$, whose exact solution we know.

In [ ]:
def f(x: float) -> float: return 0.0
def g(x: float) -> float: return -1.0

x_pts, h = np.linspace(0, 10, 50, retstep=True)
y_pts = np.zeros_like(x_pts)

# initial conditions taken from the known exact solution
y0 = 1.2 * np.cos(0.4)
y1 = 1.2 * np.cos(h + 0.4)
numerov(x_pts, y_pts, f, g, y0, y1)

fig, ax = plt.subplots()
ax.plot(x_pts, 1.2 * np.cos(x_pts + 0.4), "--", color=PALETTE[1], lw=2.5,
        label=r"exact: $1.2\cos(x+0.4)$")
ax.plot(x_pts, y_pts, "o", ms=4.5, color=PALETTE[0], label="Numerov, $N=50$")
ax.set_xlabel("$x$")
ax.set_ylabel("$y(x)$")
ax.set_title("Testing Numerov's method")
ax.legend();

The recursion above is quoted as accurate to $O(h^6)$ per step. That is a *local* truncation error; what actually matters in practice is how the *global* error, accumulated over the whole integration range, decreases as we refine the grid. Let's measure it directly, by comparing the numerical and exact solutions over a range of step sizes and fitting the result on a log-log scale.

In [ ]:
n_points = np.array([10, 20, 40, 80, 160, 320, 640])
errors = np.zeros(len(n_points), dtype=float)
h_values = np.zeros_like(errors)

for i, n in enumerate(n_points):
    x_pts, h = np.linspace(0, 10, n, retstep=True)
    y_pts = np.zeros_like(x_pts)
    y0 = 1.2 * np.cos(0.4)
    y1 = 1.2 * np.cos(h + 0.4)
    numerov(x_pts, y_pts, f, g, y0, y1)
    errors[i] = np.max(np.abs(y_pts - 1.2 * np.cos(x_pts + 0.4)))
    h_values[i] = h

slope, _ = np.polyfit(np.log(h_values), np.log(errors), 1)
print(f"measured convergence order: {slope:.2f}")

fig, ax = plt.subplots()
ax.loglog(h_values, errors, "o", ms=7, color=PALETTE[0], label="measured error")
ax.loglog(h_values, errors[0] * (h_values / h_values[0]) ** slope, "--",
          color=PALETTE[1], label=fr"$\propto h^{{{slope:.1f}}}$")
ax.set_xlabel("step size $h$")
ax.set_ylabel("max$|y_\\mathrm{num} - y_\\mathrm{exact}|$")
ax.set_title("Numerov's method: error vs. step size")
ax.legend();

The error drops close to $h^4$: even though each single step is accurate to $O(h^6)$, the errors made at each of the $\sim 1/h$ steps add up, leaving a *global* error of order $h^4$. This is still very fast convergence, halving $h$ cuts the error by a factor of about 16, and it is the reason Numerov's method is the natural choice for the eigenvalue problems below, where we will need accurate wavefunctions without using huge grids.

## Part 2: Finding bound states (the shooting method)

In the following parts, we will be interested in solving the stationary Schrödinger
equation

\begin{equation}
  -\frac{\hbar^2}{2m} \frac{d^2 \psi}{dx^2} + V(x) \psi(x) = E \psi(x)
\end{equation}

which is the same as the Numerov general equation above with

\begin{equation}
  f(x) = 0 \qquad g(x) = \frac{2m}{\hbar^2} \Big( V(x) - E \Big)
\end{equation}

It is convenient to work in units where $\hbar=1$ and $m=1$.

Let us first consider the infinite potential well, i.e. a potential
$V(x) = 0$ in the interval $[0,a]$ and $V(x) = \infty$ everywhere else.
As you remember from your quantum mechanics course, the solutions
to this problem can be chosen to be real and yield an infinite
countable number of bound states with well-defined energies:

\begin{equation}
  E_n = \frac{n^2 \hbar^2 \pi^2}{2 m a^2}
\end{equation}

How can you solve this problem numerically? It is clear that we know
the boundary conditions:

\begin{equation}
  \psi(0) = \psi(a) = 0
\end{equation}

We can use Numerov's method and solve for $\psi(x)$ starting
from $x=0$ with $\psi_0 = \psi(0) = 0$. We can then actually
choose $\psi_1 = \psi(h)$ freely because the wavefunction
that is solution to the Schrödinger equation is only defined up to a
multiplicative factor. For a randomly chosen energy $E$ what will
happen is that the solution found using Numerov's method will have
a finite value $m = \psi(a)$ at $x=a$. But we know the wavefunction
should vanish there.

In order to find the wavefunctions that satisfy the boundary
conditions you can use the **shooting method**: start from some
energy $E$ and change it until you find that the mismatch
$m$ is zero.

You can proceed in the following steps:

1. Write a function of the energy (and other relevant parameters)
   ```python
   def get_mismatch(energy, ...)
   ```
   that returns the value of the mismatch at $x=a$

2. Make a plot of this mismatch as a function of energy. What do you
   observe?

3. Find the energies for which the mismatch is zero. Basically, you
   just have to find the zeros of the `get_mismatch` function. There
   are many tools to do this. Try to find one from the
   [scipy project](http://docs.scipy.org/doc/scipy/reference).
   *Hint*: What about [`scipy.optimize.brentq`](http://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.brentq.html#scipy.optimize.brentq)?

4. Plot the wavefunction for the eigenenergies you found and see
   whether they behave like you expect (number of nodes, etc.).

5. It might be nicer to wrap the methods you have used into a
   class. Try to see if you know how this could be done.


## Solution

Here is an implementation that uses a class. It is perfectly fine to solve this problem without a class, but I want you to slowly get used to their construction.

In [ ]:
from scipy.optimize import brentq

class InfiniteWell:
    """Bound states of a particle in an infinite square well [x_min, x_max]."""

    m: float = 1.0
    hbar: float = 1.0
    eps: float = 0.1  # arbitrary nonzero slope, needed to start the recursion

    def __init__(self, n_points: int, x_min: float, x_max: float) -> None:
        self.x, self.step = np.linspace(x_min, x_max, n_points, retstep=True)
        self.y = np.zeros(n_points)

    def compute_wavefunction(self, energy: float) -> None:
        def f(x: float) -> float: return 0.0
        def g(x: float) -> float: return -2 * self.m * energy / self.hbar**2
        numerov(self.x, self.y, f, g, 0.0, self.eps)

    def get_mismatch(self, energy: float) -> float:
        self.compute_wavefunction(energy)
        return self.y[-1]

    def find_energy(self, e_min: float, e_max: float) -> float:
        return brentq(self.get_mismatch, e_min, e_max)


Now that the solver class is defined we can solve the Schrödinger equation. In order to have an idea where to look for eigenenergies, it is always a good idea to do a quick energy scan and look at the mismatch: its zeros are the eigenenergies. After this quick scan, a more refined search with a relevant energy window can be done with `brentq`.

In [ ]:
well = InfiniteWell(50, 0, 1)
e_range = np.arange(0, 100, 0.5)
mismatch = np.vectorize(well.get_mismatch)(e_range)

fig, ax = plt.subplots()
ax.axhline(0, color="#898781", lw=1)
ax.plot(e_range, mismatch, color=PALETTE[0], label="mismatch $\\psi(a)$")
for n in range(1, 5):
    ax.axvline(np.pi**2 * n**2 / 2, color=PALETTE[1], ls="--", lw=1.2, alpha=0.8)
ax.plot([], [], color=PALETTE[1], ls="--", label="exact $E_n$")  # legend entry only
ax.set_xlabel("$E$")
ax.set_ylabel("mismatch $m$")
ax.set_title("Locating the eigenenergies")
ax.legend();

The zeros line up with the analytical energies, as expected. We can now pin them down precisely and plot the corresponding wavefunctions.

In [ ]:
fig, ax = plt.subplots()
ax.axvspan(0, 1, color="#f0efec", zorder=0, label="the well")

for n, (e_min, e_max) in enumerate([(0, 10), (10, 30), (40, 50), (70, 90)], start=1):
    energy = well.find_energy(e_min, e_max)
    well.compute_wavefunction(energy)
    color = PALETTE[(n - 1) % len(PALETTE)]
    ax.axhline(energy, color="#898781", lw=0.8, ls=":")
    ax.plot(well.x, energy + 20 * well.y, color=color, label=f"$n={n}$")
    exact = np.pi**2 * n**2 / 2
    print(f"E_{n} = {energy:12.8f}   (exact: {exact:12.8f})")

ax.set_xlim(-0.2, 1.2)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$E_n + \mathrm{const} \times \psi_n(x)$")
ax.set_title("The first four bound states of the infinite well")
ax.legend(loc="upper left", fontsize=10);

## Part 3: Bound states with asymptotic boundary conditions

In the infinite potential well problem, the boundary conditions are very easy. It is enough
to ask that the wavefunction be zero at the edges of the well. However, in most cases one
is not dealing with a truly infinite potential. In the case of bound states in a generic
potential, one will usually have two side regions where the wavefunction goes to zero
as $|x| \rightarrow \infty$ and a central region with an oscillatory behavior. In that
more generic case, the boundary conditions are given at infinity (where the wavefunction
is going to zero). This is illustrated in the
plot below, where we show a potential $V(x)$ and one of the eigenstates that decays to zero as
$|x| \rightarrow \infty$:

![Example of bound state](boundstate.png)

The dashed line is the energy of the bound state. It crosses the potential $V(x)$
at the values $x_L$ and $x_R$. We can expect the wavefunction to have an oscillatory behavior
between $x_L$ and $x_R$ and to decay exponentially to zero for $x < x_L$ and
$x > x_R$.

Dealing with a condition at infinity is not very difficult. It is enough to work in a
large window $[x_\mathrm{min}, x_\mathrm{max}]$ and ask for the wavefunction to be zero
at $x_\mathrm{min}$ and $x_\mathrm{max}$. One can then increase the size of the window
to see if results change or not.

The problem is how to satisfy the boundary conditions. One could in principle use the
shooting method again: start from the left with a wavefunction that is zero at $x_\mathrm{min}$
and then propagate using Numerov's method all the way to $x_\mathrm{max}$ and tune the energy
until the wavefunction is zero at $x_\mathrm{max}$.

There is however a problem with this approach. Propagating the wavefunction from the oscillatory
region into the exponentially decaying region will lead to a large error accumulation. This is
because an exponentially increasing solution is also a possible solution of the equation and
can destroy the accuracy of the algorithm. The rule of thumb is to
avoid integrating into the exponential regions.

Instead, it is more efficient to propagate both forward from the left and backward from
the right and impose that both solutions satisfy the continuity conditions at a point
$x_0$ somewhere in the oscillatory region. In practice you can follow these steps
in your code:

1. Choose a large window (large enough that you expect the wavefunction to be
   essentially zero at the boundaries) $[x_\mathrm{min}, x_\mathrm{max}]$.

2. Integrate the solution first starting from $x_\mathrm{min}$ up to a given value $x = x_0$
within the interval $x_L \le x_0 \le x_R$. One can typically choose $x_0 = x_R$. We will
call this solution $\psi_L(x)$.

3. Integrate the solution starting from $x_\mathrm{max}$ down to $x = x_0$. We will call this
solution $\psi_R(x)$. You will need to write a modified version of the `numerov` function
that integrates the solution backward.

4. With the correct energy, the following continuity conditions should be
   satisfied at $x = x_0$, i.e.:

   \begin{align}
       \psi_R(x_0) &= \psi_L(x_0) \\
       \psi'_R(x_0) &= \psi'_L(x_0)
   \end{align}

   Both $\psi_R$ and $\psi_L$ are solutions up to a multiplicative factor. So one can always
   normalize them to be equal at $x_0$. The conditions above can then be rewritten as a
   single matching condition for the logarithmic derivative:

   \begin{equation}
       \frac{\psi'_R(x_0)}{\psi_R(x_0)} = \frac{\psi'_L(x_0)}{\psi_L(x_0)}
   \end{equation}

   Define a function of the energy (and other relevant parameters)
   ```python
   def get_mismatch(energy, ...)
   ```
   that returns the value of the logarithmic derivative mismatch at $x=x_0$.

5. Make a plot of this mismatch as a function of energy. What do you
   observe?

6. Find the energies for which the mismatch is zero.

7. Plot the wavefunctions for the eigenenergies you found and see
   whether they behave like you expect (number of nodes, etc.).

8. Again, it might be nicer to wrap the methods you have used into a
   class. Try to see if you know how this could be done.

9. Test your code for the following potential:

   \begin{equation}
      V(x) = \frac{\hbar^2}{2m} \alpha^2 \lambda (\lambda-1) \Big[ \frac{1}{2} - \frac{1}{\cosh^2 (\alpha x)} \Big]
   \end{equation}

   You can consider the case $\alpha = 1$ and $\lambda = 4$ and take units such that $\hbar = m = 1$.


## Solution

Here is a modified version of the `numerov` function that can also propagate backwards. Note how `xt[::-1]` gives a reversed *view* of the array `xt` -- no data is copied, and writing into the reversed view of `yt` updates `yt` itself, in place.

In [ ]:
def numerov(
    xt: np.ndarray,
    yt: np.ndarray,
    f: Callable[[float], float],
    g: Callable[[float], float],
    y0: float,
    y1: float,
    forward: bool = True,
) -> None:
    """Solve y'' = f(x) + g(x) y(x) with Numerov's method.

    With forward=False, the recursion runs on `xt` and `yt` read back to
    front, i.e. it integrates from the last point down to the first.
    """
    x, y = (xt, yt) if forward else (xt[::-1], yt[::-1])

    y[0], y[1] = y0, y1
    h = x[1] - x[0]
    b = h**2 / 12.0
    f_vals = np.vectorize(f)(x)
    g_vals = np.vectorize(g)(x)

    for i in range(1, len(y) - 1):
        y[i + 1] = (
            2 * y[i] - y[i - 1]
            + b * (f_vals[i + 1] + 10 * (f_vals[i] + g_vals[i] * y[i])
                   + f_vals[i - 1] + g_vals[i - 1] * y[i - 1])
        ) / (1 - g_vals[i + 1] * b)


I now write a `BoundState` class that resembles very much the one we had for the infinite well.

In [ ]:
class BoundState:
    """Bound states of a particle in a generic potential that vanishes (or
    plateaus) as |x| -> infinity, found by matching the solutions
    propagated inward from the two edges of a finite window.
    """

    m: float = 1.0
    hbar: float = 1.0
    eps: float = 1e-3

    def __init__(self, potential: Callable[[float], float], n_points: int,
                 x_min: float, x_max: float) -> None:
        self.x, self.step = np.linspace(x_min, x_max, n_points, retstep=True)
        self.y = np.zeros(n_points)
        self.V = potential

    def get_match_index(self, energy: float) -> int:
        """Index of the grid point closest to where V(x) crosses `energy`."""
        for i in range(len(self.x) - 1):
            if (self.V(self.x[i]) - energy) * (self.V(self.x[i + 1]) - energy) < 0:
                return i
        return 0

    def get_mismatch(self, energy: float) -> float:
        def f(x: float) -> float: return 0.0
        def g(x: float) -> float: return 2 * self.m * (self.V(x) - energy) / self.hbar**2

        n_match = self.get_match_index(energy)  # index of the matching point x_0

        # propagate the solution from the left
        numerov(self.x[: n_match + 2], self.y[: n_match + 2], f, g, 0.0, self.eps, forward=True)
        self.y[: n_match + 2] /= self.y[n_match]
        log_deriv_left = self.y[n_match + 1] - self.y[n_match - 1]

        # propagate the solution from the right
        numerov(self.x[n_match - 1:], self.y[n_match - 1:], f, g, 0.0, self.eps, forward=False)
        self.y[n_match - 1:] /= self.y[n_match]
        log_deriv_right = self.y[n_match + 1] - self.y[n_match - 1]

        return log_deriv_left - log_deriv_right

    def find_energy(self, e_min: float, e_max: float) -> float:
        return brentq(self.get_mismatch, e_min, e_max)


Now that the solver class is defined we can introduce a potential and solve the Schrödinger equation. As before, a quick energy scan tells us where to look for eigenenergies before a more refined search with `brentq`.

The potential I use below is a classic in quantum mechanics, a reflectionless, or Pöschl-Teller, potential well, of the same family that shows up as a soliton solution of the Korteweg-de Vries equation.

In [ ]:
def potential(x: float) -> float:
    alpha, lam = 1.0, 4.0
    return 0.5 * alpha**2 * lam * (lam - 1) * (0.5 - 1 / np.cosh(alpha * x) ** 2)

boundstate = BoundState(potential, 501, -10, 10)
e_range = np.arange(-2.99, 2.99, 0.1)
mismatch = np.vectorize(boundstate.get_mismatch)(e_range)

fig, ax = plt.subplots()
ax.axhline(0, color="#898781", lw=1)
ax.plot(e_range, mismatch, color=PALETTE[0])
ax.set_ylim(-1, 1)
ax.set_xlabel("$E$")
ax.set_ylabel("mismatch")
ax.set_title("Locating the eigenenergies");

The plot above shows clearly that there are three bound states. Let's pin down the energies more precisely and plot the wavefunctions.

In [ ]:
fig, ax = plt.subplots()
ax.plot(boundstate.x, np.vectorize(boundstate.V)(boundstate.x), color="#0b0b0b",
        lw=1.5, label="$V(x)$")

for n, (e_min, e_max) in enumerate([(-2, -1.2), (0, 1.1), (2, 2.51)]):
    energy = boundstate.find_energy(e_min, e_max)
    boundstate.get_mismatch(energy)
    ax.plot(boundstate.x, boundstate.y, color=PALETTE[n], label=f"$n={n}$, $E={energy:.3f}$")
    print(f"E_{n} = {energy:12.8f}")

ax.set_xlim(-5, 5)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\psi(x)$, $V(x)$")
ax.set_title("Bound states of a reflectionless well")
ax.legend(loc="upper right", fontsize=10);

As expected, the ground state has no node, the first excited state has one, and the second excited state has two. This is exactly the pattern you know from the harmonic oscillator or the infinite well.

## Part 4: Quantum scattering

Congratulations if you made it here! This last part is not compulsory, I have put it here
for completeness. You can try to solve it if you have time.

Here, we want to focus on scattering states, namely states that have an energy higher than
the asymptotic value of the potential. They are quite different from the bound states
we studied above for two reasons. First, there is no quantization of the energy, we will
not have to find eigenenergies. Second, the wavefunction is a complex function. It cannot
always be made real like for bound states.

We will suppose that the potential is zero except in some central region $[x_\mathrm{min}, x_\mathrm{max}]$.
Here is an example of such a potential:

![Scattering potential](scattering.png)

The black line is the potential and the blue and orange curves are the real and imaginary part
of a given scattering state (here with energy $E=0.7$).
We know the expression for the wavefunction on the left $\psi_L(x)$ and on the right $\psi_R(x)$
is given by

\begin{equation}
  \psi_L(x) = e^{ikx} + A e^{-ikx} \qquad \psi_R(x) = B e^{ikx}
\end{equation}

If the energy is given by $E$, then we have

\begin{equation}
  k^2 = \frac{2 m E}{\hbar^2}
\end{equation}

Note that we explicitly assumed that the scattering state travels from the left to
the right (there is no wave coming back from the right). We also chose to normalize
the wavefunction such that the first prefactor of $\psi_L$ is 1.

Given the energy $E$ we need to find the complex coefficients $A$ and $B$. This can
be done following these steps:

1. Start with a guess $A_r$ for $A$.
2. Propagate the solution from $x_\mathrm{min}$ to $x_\mathrm{max}$. You can
   find the initial conditions analytically. Let's call $\psi_{0r}$ the value of the
   wavefunction at $x_\mathrm{min}$.
3. From the obtained solution at $x_\mathrm{max}$, deduce the value of $B$.
4. With this value of $B$ and your knowledge of $\psi_R(x_\mathrm{max})$
   and $\psi'_R(x_\mathrm{max})$ propagate the solution back to
   $x_\mathrm{min}$.
5. This solution at $x_\mathrm{min}$ has a value $\psi_{0l}$.
6. Of course, the correct $A$ is such that $\psi_{0r} = \psi_{0l}$. So you can define
   a mismatch function

   ```python
   def get_mismatch(a_coef)
   ```

   that returns $|\psi_{0r}-\psi_{0l}|^2$. Here `a_coef` is a tuple `[A_r_real, A_r_imag]`
   made of the real and imaginary part of $A_r$. Instead of finding the zero
   of `get_mismatch`, you can try to use a minimization function. Finding a minimum
   for a function of several variables is often more efficient. You can use
   the function `scipy.optimize.minimize`. Of course one should check that the minimum
   value that is found is indeed 0.
7. You can then check your code for the potential depicted above, namely the function

   \begin{equation}
     V(x) = 1 \quad \text{if} \quad 0 < x < 1 \quad \text{or} \quad 3 < x < 5
     \quad \text{otherwise} \quad V(x) = 0
   \end{equation}
8. Plot the wavefunction for a chosen energy, e.g. $E = 0.7$.
9. Plot the logarithm of the transmission $T = 1-|A|^2$ for energies in the
   interval $[0,2]$. What do you observe at $E \simeq 0.4$?
10. You might have noticed that the code does take some time to run. How could it be
    optimized?


## Solution

Here is a class `Scattering` that implements the steps described above.

In [ ]:
from scipy.optimize import minimize

class Scattering:
    """Scattering state of the 1D Schrodinger equation through a potential
    that vanishes outside a compact region [x_min, x_max].
    """

    m: float = 1.0
    hbar: float = 1.0

    def __init__(self, potential: Callable[[float], float], energy: float,
                 n_points: int, x_min: float, x_max: float) -> None:
        self.x_min, self.x_max = x_min, x_max
        self.x, self.step = np.linspace(x_min, x_max, n_points, retstep=True)
        self.y = np.zeros(n_points, dtype=complex)  # note the complex dtype
        self.V = potential
        self.energy = energy
        self.k = np.sqrt(2 * self.m * energy) / self.hbar

    def get_mismatch(self, a_coef: tuple[float, float]) -> float:
        def f(x: float) -> float: return 0.0
        def g(x: float) -> float: return 2 * self.m * (self.V(x) - self.energy) / self.hbar**2

        xl, xr, k = self.x_min, self.x_max, self.k
        A = a_coef[0] + 1j * a_coef[1]

        yl0 = np.exp(1j * k * xl) + A * np.exp(-1j * k * xl)
        yl1 = yl0 + self.step * 1j * k * (np.exp(1j * k * xl) - A * np.exp(-1j * k * xl))
        numerov(self.x, self.y, f, g, yl0, yl1, forward=True)

        yr0 = self.y[-1]
        B = np.exp(-1j * k * xr) * yr0
        yr1 = yr0 - self.step * 1j * k * B * np.exp(1j * k * xr)
        numerov(self.x, self.y, f, g, yr0, yr1, forward=False)

        return abs(self.y[0] - yl0) ** 2

    def find_coef(self, a_guess: complex = 0.01 + 0.01j) -> complex:
        result = minimize(self.get_mismatch, [a_guess.real, a_guess.imag])
        return result.x[0] + 1j * result.x[1]


Let's consider the potential depicted above and compute the corresponding scattering state.

In [ ]:
energy = 0.70

def potential(x: float) -> float:
    return 1.0 if (0 < x < 1 or 3 < x < 5) else 0.0

scatter = Scattering(potential, energy, 501, -3, 8)
A = scatter.find_coef()
scatter.get_mismatch([A.real, A.imag])

fig, ax = plt.subplots()
ax.plot(scatter.x, np.vectorize(scatter.V)(scatter.x), color="#0b0b0b", lw=2.2, label="$V(x)$")
ax.plot(scatter.x, scatter.y.real, color=PALETTE[0], label=r"$\mathrm{Re}\,\psi(x)$")
ax.plot(scatter.x, scatter.y.imag, color=PALETTE[1], label=r"$\mathrm{Im}\,\psi(x)$")
ax.set_xlabel("$x$")
ax.set_ylabel(r"$\psi(x)$")
ax.set_title(f"Scattering state at $E={energy}$")
ax.legend(loc="upper right", fontsize=10)

print(f"transmission = {1.0 - abs(A) ** 2:.4f}")


We can now compute the transmission coefficient as a function of the incoming energy.

In [ ]:
e_range = np.concatenate([np.arange(0.02, 0.7, 0.02), np.arange(0.7, 2.0, 0.1)])
log_trans = np.zeros_like(e_range)

for i, en in enumerate(e_range):
    scatter = Scattering(potential, en, 101, 0.0, 5.0)
    A = scatter.find_coef()
    log_trans[i] = np.log10(1 - abs(A) ** 2)

below_barrier = e_range < 1.0  # the barrier height is V=1; look for the
i_peak = np.flatnonzero(below_barrier)[np.argmax(log_trans[below_barrier])]  # sub-barrier resonance

fig, ax = plt.subplots()
ax.plot(e_range, log_trans, "o-", ms=4.5, color=PALETTE[0])
ax.annotate(
    "resonance",
    xy=(e_range[i_peak], log_trans[i_peak]),
    xytext=(e_range[i_peak] + 0.35, log_trans[i_peak] - 1.3),
    arrowprops=dict(arrowstyle="->", color="#898781"),
    fontsize=11,
)
ax.set_xlabel("$E$")
ax.set_ylabel(r"$\log_{10} T$")
ax.set_title("Transmission versus energy");

You can see that the transmission has a peak at $E \simeq 0.4$, well below the height of the potential barriers. This is a resonance: the well is momentarily supporting something close to a quasi-bound state between the two barriers, and the incoming wave couples to it almost perfectly. You could plot the corresponding wavefunction to see the amplitude build up between the barriers.

### A quick look at performance

The scan above calls `find_coef` once per energy; each call runs a minimization that itself calls `numerov` several times, and every call to `numerov` re-evaluates `f` and `g` on the whole grid with `np.vectorize`. `np.vectorize` is convenient because it lets `numerov` accept arbitrary scalar functions, but it is really a Python loop wearing a numpy costume. For `Scattering`, this is wasted work: the potential and the energy do not change between calls to `get_mismatch`, so $g(x) = 2m(V(x)-E)/\hbar^2$ can be computed once per scattering problem instead of twice per call.

Let's measure how much this matters, by rewriting `numerov` to accept `f` and `g` already evaluated on the grid, and timing both versions on the transmission scan.

In [ ]:
import timeit

def numerov_fast(x, y, f_vals, g_vals, y0, y1, forward=True):
    """Same recursion as `numerov`, but taking f and g already evaluated on
    the grid, rather than the scalar callables f, g."""
    if not forward:
        x, y, f_vals, g_vals = x[::-1], y[::-1], f_vals[::-1], g_vals[::-1]

    y[0], y[1] = y0, y1
    h = x[1] - x[0]
    b = h**2 / 12.0
    for i in range(1, len(y) - 1):
        y[i + 1] = (
            2 * y[i] - y[i - 1]
            + b * (f_vals[i + 1] + 10 * (f_vals[i] + g_vals[i] * y[i])
                   + f_vals[i - 1] + g_vals[i - 1] * y[i - 1])
        ) / (1 - g_vals[i + 1] * b)


class FastScattering(Scattering):
    """Same physics as `Scattering`, but V and the energy are evaluated once,
    in __init__, instead of at every call to `get_mismatch`."""

    def __init__(self, potential, energy, n_points, x_min, x_max):
        super().__init__(potential, energy, n_points, x_min, x_max)
        self.f_vals = np.zeros(n_points)
        self.g_vals = 2 * self.m * (np.vectorize(potential)(self.x) - energy) / self.hbar**2

    def get_mismatch(self, a_coef):
        xl, xr, k = self.x_min, self.x_max, self.k
        A = a_coef[0] + 1j * a_coef[1]

        yl0 = np.exp(1j * k * xl) + A * np.exp(-1j * k * xl)
        yl1 = yl0 + self.step * 1j * k * (np.exp(1j * k * xl) - A * np.exp(-1j * k * xl))
        numerov_fast(self.x, self.y, self.f_vals, self.g_vals, yl0, yl1, forward=True)

        yr0 = self.y[-1]
        B = np.exp(-1j * k * xr) * yr0
        yr1 = yr0 - self.step * 1j * k * B * np.exp(1j * k * xr)
        numerov_fast(self.x, self.y, self.f_vals, self.g_vals, yr0, yr1, forward=False)

        return abs(self.y[0] - yl0) ** 2


def run_scan(scattering_cls, n_energies=30):
    for en in np.linspace(0.02, 2.0, n_energies):
        scattering_cls(potential, en, 101, 0.0, 5.0).find_coef()

t_original = timeit.timeit(lambda: run_scan(Scattering), number=1)
t_fast = timeit.timeit(lambda: run_scan(FastScattering), number=1)

print(f"original  Scattering: {t_original:.2f} s")
print(f"optimized Scattering: {t_fast:.2f} s  ({t_original / t_fast:.1f}x faster)")


Removing the redundant `np.vectorize` calls buys us a real but modest speed-up (around 30-40% here, and it does not depend much on the grid size `N`). That is worth having for free, but it tells us the rest of the time is spent elsewhere: most of it is inside `scipy.optimize.minimize` itself, which by default estimates gradients by finite differences, i.e. by calling `get_mismatch` several more times per iteration. The recursion in `numerov`, `y[i+1] = ...` depending on `y[i]` and `y[i-1]`, cannot be vectorized over `i` either way, since each point genuinely depends on the two previous ones; for a much finer grid, that loop is exactly what you would move to `numba` or Cython rather than trying to out-clever it in numpy. The general lesson holds beyond this example: guess where the time goes, then measure it, because the two rarely agree completely.